# Дообучение BIOT на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-21 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить BIOT (NeurIPS 2023, github.com/ycq091044/BIOT, MIT) по той же схеме, что REVE, CBraMod и DeepEpilepsy, и сравнить / собрать в ансамбль |
| **Веса** | `EEG-PREST-16-channels.ckpt` — предобучение только на рутинных ЭЭГ MGH (5 млн окон), **TUH не видела**. Варианты на 18 каналов не нужны: два лишних канала C3-A2/C4-A1 нужны для сна SHHS, а «six datasets» видела обучающие части TUAB/TUEV |
| **Данные** | Обучение: NMT train (`mhmdrdwn/nmt-eeg`), до 8 окон на запись, 15 % записей — выбор скорости обучения и эпохи. Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`) |
| **Вход** | Как у авторов: 200 Гц, окна 10 с, 16 биполярных пар (порядок из README и `datasets/TUAB/process.py`), каждый канал делится на 95-й процентиль модуля (абсолютная амплитуда и соотношение каналов по амплитуде пропадают). Пары считаются вычитанием из 19 каналов 10-20. Фильтр 0,3–75 Гц + режекция 50 Гц — как в остальных блокнотах |
| **Аугментации** | шум 0,1, зануление 1–2 каналов (p = 0,5), зеркальная замена полушарий (p = 0,5), сдвиг до 2 с |
| **Обучение** | вся модель (3,2 млн параметров), голова авторов (ELU + линейный слой), AdamW, косинус, пакет 32, fp32 (STFT с n_fft = 200 в fp16 не работает); скорость основы выбирается по отложенным 15 % из 1·10⁻⁵ / 1·10⁻⁴ / 1·10⁻³ (голова 1·10⁻³), затем 3 запуска |
| **Среда** | Kaggle, GPU T4 |

In [ ]:
import os, sys, glob, time, subprocess, warnings
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # версии зависимостей 2021–2023 г.: с ними веса авторов грузятся без расхождений ключей (проверено локально)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "linear_attention_transformer==0.19.1",
                    "local-attention==1.8.6", "linformer==0.2.1", "product-key-memory==0.1.10", "axial-positional-embedding==0.2.1"], check=True)
    if not os.path.exists("/kaggle/working/BIOT"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ycq091044/BIOT.git", "/kaggle/working/BIOT"], check=True)
    BIOT_DIR = "/kaggle/working/BIOT"
else:
    BIOT_DIR = os.environ["BIOT_DIR"]
sys.path.insert(0, BIOT_DIR)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from model.biot import BIOTEncoder
warnings.filterwarnings("ignore", message=".*window was not provided.*")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_NB = time.time()

SFREQ, SEG = 200, 10
L = SFREQ * SEG
CAP_EVAL = 60          # окон на запись для NMT eval и Турции (равномерно по записи), как у REVE

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
# 16 пар в порядке предобучения PREST (README; T7/T8/P7/P8 = T3/T4/T5/T6). Fz, Cz, Pz не используются
PAIRS = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"), ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),
         ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"), ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")]
PA = [EL.index(a) for a, _ in PAIRS]; PB = [EL.index(b) for _, b in PAIRS]
NCH = len(PAIRS)
MIRROR = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11]   # левые цепочки <-> правые
CKPT = f"{BIOT_DIR}/pretrained-models/EEG-PREST-16-channels.ckpt"

def bt_input(W):
    """19 каналов -> 16 биполярных пар -> каждый канал / 95-й процентиль модуля (как в utils.py авторов)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    x = x[:, PA] - x[:, PB]
    return x / (torch.quantile(x.abs(), 0.95, dim=-1, keepdim=True) + 1e-8)

class BTNet(nn.Module):
    """Кодировщик BIOT (веса PREST) + голова авторов ClassificationHead (ELU -> Linear) на 1 выход."""
    def __init__(self):
        super().__init__()
        self.biot = BIOTEncoder(emb_size=256, heads=8, depth=4, n_channels=16, n_fft=200, hop_length=100)
        res = self.biot.load_state_dict(torch.load(CKPT, map_location="cpu"))
        assert not res.missing_keys and not res.unexpected_keys, res
        self.head = nn.Sequential(nn.ELU(), nn.Linear(256, 1))
    def forward(self, x):
        return self.head(self.biot(x)).squeeze(-1)

def new_bt(seed):
    torch.manual_seed(seed); np.random.seed(seed)   # np: внутри BIOT perturb не используется, но на всякий случай
    return BTNet().to(DEV)

def predict(model, Ws, bs=128):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad():
        for W in Ws:
            p = []
            for b in range(0, len(W), bs):
                p.append(torch.sigmoid(model(bt_input(W[b:b + bs]))).cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3-75 Гц + режекция 50 Гц, как в блокнотах CBraMod и REVE."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор",
      "| параметров:", sum(p.numel() for p in BTNet().parameters()))

## 2. NMT

In [ ]:
import mne
mne.set_log_level("ERROR")
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
print(pd.crosstab(lab["loc"], lab["label"]))
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
# подкачка обоих наборов параллельно
t0 = time.time(); allf = list(lab.path) + [f for f, _ in tfiles]
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, allf)):
        if (i + 1) % 500 == 0 or i + 1 == len(allf):
            print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

NMT_STORE_K = 100.0   # масштаб в файлах неверный; каналы всё равно нормируются, множитель только для float16
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
    except Exception:
        return r.recordname, None
    W = windows(cb_filter(X))
    if len(W) == 0:
        return r.recordname, None
    if r["loc"] == "train" and len(W) > 8:
        W = W[np.sort(np.random.default_rng(i).choice(len(W), 8, replace=False))]
    elif r["loc"] == "eval":
        W = W[spread(len(W), CAP_EVAL)]
    return r.recordname, W.astype(np.float16)

NMT = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон: train", sum(len(NMT[n]) for n in tr.recordname), "| eval", sum(len(NMT[n]) for n in ev.recordname))

## 3. Турецкий набор: первые 19 каналов (относительно уха), 500→200 Гц, куски по 15 с подряд, окна 10 с

In [ ]:
# индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)          # средний референт (для биполярных пар не важен), как в образце — для СКО записи
    X = ss.resample_poly(X, 2, 5, axis=-1)    # 200 Гц
    W = windows(cb_filter(np.concatenate(list(X), axis=-1)))
    return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))

TR, TR_AMP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
        TR.append(a); TR_AMP.append(b)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR]))}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение BIOT

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
Xw = np.concatenate([NMT[n] for n in fit_names])
val_W = [NMT[n] for n in val_names]
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)}")

In [ ]:
NOISE, BS, EPOCHS, LR_HEAD = 0.1, 32, 6, 1e-3
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()

def augment(x, g):
    B = x.shape[0]
    x = x + NOISE * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(NCH, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

def make_opt(model, total_steps, lr_back):
    back = [p for n, p in model.named_parameters() if n.startswith("biot") and p.requires_grad]
    head = [p for n, p in model.named_parameters() if n.startswith("head")]
    opt = torch.optim.AdamW([{"params": back, "lr": lr_back}, {"params": head, "lr": LR_HEAD}], weight_decay=0.05)
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, x, t):
    out = model(x)
    loss = F.binary_cross_entropy_with_logits(out, t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    sched.step()
    return loss.item(), bad

def train_run(seed, lr_back, epochs=EPOCHS, tag="biot"):
    g = np.random.default_rng(seed)
    model = new_bt(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, epochs * steps_ep, lr_back)
    best_auc, best_state, best_ep, hist, n_bad = -1, None, 0, [], 0
    t0 = time.time()
    for ep in range(epochs):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            l, bad = step(model, opt, sched, augment(bt_input(Xw[j]), g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(auc, 3)))
        print(f"  {tag} lr {lr_back:g} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_ep, best_auc, hist, n_bad

In [ ]:
# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны убывать и уйти ниже 0,69 (64 шага, жёсткого порога нет: у BIOT на реальных данных ~0,6) ---
g0 = np.random.default_rng(123)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], 128, replace=False) for c in (0, 1)])
model = new_bt(0); opt, sched = make_opt(model, 8 * (len(sm) // BS), 1e-3)
model.train(); sm_losses, tt = [], []
for ep in range(8):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, bt_input(Xw[j]), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:]))
del model, opt
if DEV == "cuda": torch.cuda.empty_cache()
assert np.isfinite(sm_losses).all() and np.mean(sm_losses[-2:]) < np.mean(sm_losses[:2]) - 0.05 and min(sm_losses) < 0.66, f"нет сходимости: {sm_losses}"
est_run = (EPOCHS * (len(yw) / BS) * t_step * 1.5 + EPOCHS * 20) / 60   # мин (+50 % на аугментации, ~20 с на выбор эпохи)
print(f"шаг {t_step:.3f} с | оценка одного запуска {est_run:.1f} мин")

In [ ]:
# --- выбор скорости обучения основы: только по отложенным 15 % NMT train (запуск 0) ---
TIME_LIMIT_MIN = 40   # бюджет задачи 1,5 ч GPU; чтение данных ~20 мин
T_TRAIN, GRID = time.time(), {}
for lr in [1e-5, 1e-4, 1e-3]:
    if (time.time() - T_TRAIN) / 60 + est_run > TIME_LIMIT_MIN / 2:
        print(f"ПРОПУСК lr {lr:g}: не укладывается"); continue
    model, best_ep, best_auc, hist, n_bad = train_run(0, lr, tag="выбор")
    GRID[lr] = dict(model=model.cpu(), best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad)
    print(f"lr {lr:g}: лучшая эпоха {best_ep}, отложенные NMT {best_auc:.3f}", flush=True)
LR = max(GRID, key=lambda k: GRID[k]["best_auc"])
print("выбрана скорость основы:", LR)
pd.DataFrame([dict(lr_back=k, best_epoch=v["best_ep"], val=round(v["best_auc"], 3), hist=v["hist"], nan_steps=v["n_bad"]) for k, v in GRID.items()]) \
  .to_csv("/kaggle/working/biot_lr_choice.csv" if ON_KAGGLE else "biot_lr_choice.csv", index=False)

# --- 3 запуска с выбранной скоростью (запуск 0 — уже обученный при выборе); тестовые наборы — один раз ---
RUNS, dur = [], []
for seed in [0, 1, 2]:
    t0 = time.time()
    if seed == 0:
        G = GRID[LR]; model, best_ep, best_auc, hist, n_bad = G["model"].to(DEV), G["best_ep"], G["best_auc"], G["hist"], G["n_bad"]
    else:
        est = np.mean(dur) if dur else est_run * 60
        if (time.time() - T_TRAIN + est) / 60 > TIME_LIMIT_MIN:
            print(f"ПРОПУСК запуск {seed}: не укладывается в {TIME_LIMIT_MIN} мин"); continue
        model, best_ep, best_auc, hist, n_bad = train_run(seed, LR)
    s_ev = predict(model, [NMT[n] for n in ev.recordname]); s_tr = predict(model, TR)
    dur.append(time.time() - t0)
    RUNS.append(dict(model="biot", lr_back=LR, seed=seed, best_epoch=best_ep, val=round(best_auc, 3), hist=hist, nan_steps=n_bad,
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr)), s_val=predict(model, val_W)))
    print(f"biot запуск {seed}: эпоха {best_ep} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
          f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)}", flush=True)
    del model
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RUNS])
runs.to_csv("/kaggle/working/biot_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "BIOT дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "BIOT дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv("/kaggle/working/biot_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))
d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
for d, k in [(d1, "s_ev"), (d2, "s_tr")]:
    for r in RUNS: d[f"biot_seed{r['seed']}"] = r[k]
d1["biot_ens_rank"], d2["biot_ens_rank"] = ens_ev, ens_tr
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_names), "y": y_val})
for r in RUNS: d3[f"biot_seed{r['seed']}"] = r["s_val"]
pd.concat([d1, d2, d3]).to_csv("/kaggle/working/biot_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (без строк о подкачке файлов и служебных предупреждений)

Первый запуск остановился на проверке сходимости (порог 0,55 для 64 шагов оказался слишком жёстким: потери 0,81 → 0,60 шли вниз); проверку заменил на «потери убывают и ниже 0,66», лимиты времени уменьшил. Тестовые наборы считались один раз, в итоговом запуске.

```
устройство: Tesla T4 | параметров: 3186449
label  abnormal  normal
loc
eval         90      95
train       325    1907
label  abnormal  normal
loc
eval         90      95
train       325    1904
окон: train 17775 | eval 10406
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор: 335
  проверка: эпоха 1 потери 0.808
  проверка: эпоха 2 потери 0.715
  проверка: эпоха 3 потери 0.755
  проверка: эпоха 4 потери 0.715
  проверка: эпоха 5 потери 0.652
  проверка: эпоха 6 потери 0.625
  проверка: эпоха 7 потери 0.605
  проверка: эпоха 8 потери 0.596
шаг 0.074 с | оценка одного запуска 7.2 мин
  выбор lr 1e-05 запуск 0 | эпоха 1: потери 0.611 | отложенные NMT 0.731 | шагов с NaN/inf 0 | 41 с
  выбор lr 1e-05 запуск 0 | эпоха 2: потери 0.572 | отложенные NMT 0.706 | шагов с NaN/inf 0 | 83 с
  выбор lr 1e-05 запуск 0 | эпоха 3: потери 0.554 | отложенные NMT 0.718 | шагов с NaN/inf 0 | 129 с
  выбор lr 1e-05 запуск 0 | эпоха 4: потери 0.541 | отложенные NMT 0.713 | шагов с NaN/inf 0 | 176 с
  выбор lr 1e-05 запуск 0 | эпоха 5: потери 0.533 | отложенные NMT 0.700 | шагов с NaN/inf 0 | 222 с
  выбор lr 1e-05 запуск 0 | эпоха 6: потери 0.521 | отложенные NMT 0.705 | шагов с NaN/inf 0 | 269 с
lr 1e-05: лучшая эпоха 1, отложенные NMT 0.731
  выбор lr 0.0001 запуск 0 | эпоха 1: потери 0.597 | отложенные NMT 0.788 | шагов с NaN/inf 0 | 47 с
  выбор lr 0.0001 запуск 0 | эпоха 2: потери 0.543 | отложенные NMT 0.764 | шагов с NaN/inf 0 | 94 с
  выбор lr 0.0001 запуск 0 | эпоха 3: потери 0.514 | отложенные NMT 0.805 | шагов с NaN/inf 0 | 140 с
  выбор lr 0.0001 запуск 0 | эпоха 4: потери 0.476 | отложенные NMT 0.786 | шагов с NaN/inf 0 | 187 с
  выбор lr 0.0001 запуск 0 | эпоха 5: потери 0.444 | отложенные NMT 0.782 | шагов с NaN/inf 0 | 234 с
  выбор lr 0.0001 запуск 0 | эпоха 6: потери 0.417 | отложенные NMT 0.775 | шагов с NaN/inf 0 | 281 с
lr 0.0001: лучшая эпоха 3, отложенные NMT 0.805
  выбор lr 0.001 запуск 0 | эпоха 1: потери 0.644 | отложенные NMT 0.773 | шагов с NaN/inf 0 | 47 с
  выбор lr 0.001 запуск 0 | эпоха 2: потери 0.609 | отложенные NMT 0.805 | шагов с NaN/inf 0 | 93 с
  выбор lr 0.001 запуск 0 | эпоха 3: потери 0.583 | отложенные NMT 0.784 | шагов с NaN/inf 0 | 140 с
  выбор lr 0.001 запуск 0 | эпоха 4: потери 0.564 | отложенные NMT 0.786 | шагов с NaN/inf 0 | 187 с
  выбор lr 0.001 запуск 0 | эпоха 5: потери 0.535 | отложенные NMT 0.775 | шагов с NaN/inf 0 | 234 с
  выбор lr 0.001 запуск 0 | эпоха 6: потери 0.508 | отложенные NMT 0.781 | шагов с NaN/inf 0 | 281 с
lr 0.001: лучшая эпоха 2, отложенные NMT 0.805
выбрана скорость основы: 0.0001
biot запуск 0: эпоха 3 | NMT eval 0.772 | Турция 0.786 | медиана p здоровые/эпилепсия 0.054/0.153 | разных значений 119/121
  biot lr 0.0001 запуск 1 | эпоха 1: потери 0.601 | отложенные NMT 0.725 | шагов с NaN/inf 0 | 47 с
  biot lr 0.0001 запуск 1 | эпоха 2: потери 0.543 | отложенные NMT 0.786 | шагов с NaN/inf 0 | 93 с
  biot lr 0.0001 запуск 1 | эпоха 3: потери 0.499 | отложенные NMT 0.770 | шагов с NaN/inf 0 | 140 с
  biot lr 0.0001 запуск 1 | эпоха 4: потери 0.475 | отложенные NMT 0.785 | шагов с NaN/inf 0 | 187 с
  biot lr 0.0001 запуск 1 | эпоха 5: потери 0.427 | отложенные NMT 0.762 | шагов с NaN/inf 0 | 234 с
  biot lr 0.0001 запуск 1 | эпоха 6: потери 0.426 | отложенные NMT 0.765 | шагов с NaN/inf 0 | 281 с
biot запуск 1: эпоха 2 | NMT eval 0.837 | Турция 0.801 | медиана p здоровые/эпилепсия 0.084/0.147 | разных значений 119/121
  biot lr 0.0001 запуск 2 | эпоха 1: потери 0.598 | отложенные NMT 0.783 | шагов с NaN/inf 0 | 47 с
  biot lr 0.0001 запуск 2 | эпоха 2: потери 0.545 | отложенные NMT 0.773 | шагов с NaN/inf 0 | 93 с
  biot lr 0.0001 запуск 2 | эпоха 3: потери 0.511 | отложенные NMT 0.762 | шагов с NaN/inf 0 | 140 с
  biot lr 0.0001 запуск 2 | эпоха 4: потери 0.470 | отложенные NMT 0.783 | шагов с NaN/inf 0 | 187 с
  biot lr 0.0001 запуск 2 | эпоха 5: потери 0.435 | отложенные NMT 0.746 | шагов с NaN/inf 0 | 234 с
  biot lr 0.0001 запуск 2 | эпоха 6: потери 0.430 | отложенные NMT 0.753 | шагов с NaN/inf 0 | 280 с
biot запуск 2: эпоха 4 | NMT eval 0.711 | Турция 0.812 | медиана p здоровые/эпилепсия 0.015/0.079 | разных значений 119/121
всё обучение: 24.3 мин
model  lr_back  seed  best_epoch   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
 biot      0.0     0           3 0.805          0     0.772   0.786          0.054          0.153          119
 biot      0.0     1           2 0.786          0     0.837   0.801          0.084          0.147          119
 biot      0.0     2           4 0.783          0     0.711   0.812          0.015          0.079          119
                             модель n          NMT eval                             Турция
          BIOT дообученная, запуски 3     0.774 ± 0.063 0.800 ± 0.013 (0.79 / 0.80 / 0.81)
BIOT дообученная, ансамбль запусков   0.796 (0.73-0.86)                  0.821 (0.73-0.89)
Связь с амплитудой записи на Турции (Спирмен): 0.25
всё время блокнота: 53.7 мин
```

## 7. Сравнение с другими моделями (по людям, `results/biot_ensembles_summary.csv`)

Ранги пересчитаны на общих записях (NMT eval 184 + Турция 121), поэтому у BIOT 0,798 вместо 0,796.

| Модель / ансамбль (средние ранги) | NMT eval (95% ДИ) | Турция (95% ДИ) |
|---|---|---|
| BIOT дообученная (3 запуска) | 0,798 (0,73–0,86) | 0,821 (0,73–0,89) |
| REVE | 0,879 (0,83–0,92) | **0,874 (0,80–0,93)** |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| DeepEpilepsy | **0,900 (0,85–0,94)** | 0,682 (0,59–0,77) |
| BIOT + REVE | 0,862 (0,81–0,91) | 0,876 (0,81–0,94) |
| BIOT + CBraMod | 0,839 (0,78–0,89) | 0,856 (0,78–0,92) |
| BIOT + DeepEpilepsy | 0,882 (0,83–0,93) | 0,803 (0,72–0,88) |
| BIOT + REVE + DeepEpilepsy | 0,894 (0,85–0,93) | 0,861 (0,79–0,92) |
| REVE + CBraMod + DeepEpilepsy | 0,896 (0,85–0,93) | 0,856 (0,78–0,92) |
| Все четыре | 0,888 (0,84–0,93) | 0,871 (0,80–0,93) |

Запуски BIOT: NMT eval 0,77 / 0,84 / 0,71 (0,774 ± 0,063), Турция 0,79 / 0,80 / 0,81 (0,800 ± 0,013). Скорость основы 1·10⁻⁴ выбрана по отложенным 15 % (0,805; у 1·10⁻³ тоже 0,805 — ничья, взят первый), 1·10⁻⁵ хуже (0,731).

Парный бутстреп разности ROC AUC (95% ДИ):
- BIOT − REVE: NMT eval **−0,081 (−0,137…−0,027)**; Турция −0,053 (−0,132…+0,014) — на Турции не доказано.
- BIOT − CBraMod: NMT eval −0,048 (−0,099…+0,001); Турция +0,001 (−0,082…+0,082) — равны.
- BIOT + REVE − REVE: NMT eval −0,017, Турция +0,002 — BIOT к REVE ничего не добавляет.
- Все четыре − (REVE + CBraMod + DeepEpilepsy): NMT eval −0,008, Турция +0,016 (−0,008…+0,040) — не значимо.

Согласие оценок на Турции (Спирмен): BIOT–REVE 0,66, BIOT–CBraMod 0,62, BIOT–DeepEpilepsy 0,37. Связь оценки BIOT с амплитудой записи на Турции: **+0,25** (REVE +0,31, CBraMod +0,12; одна амплитуда даёт AUC 0,76).

## 8. Пороговые метрики (`results/biot_threshold_metrics.csv`)

Оценка модели — среднее вероятностей трёх запусков. Порог BIOT — по отложенным 15 % NMT train (максимум индекса Юдена: 0,472; для чувствительности 80 %: 0,304), на Турцию перенесён тем же. **Для REVE, CBraMod, DeepEpilepsy вероятностей на валидации нет, порог взят по самой NMT eval (оптимистично для строки NMT eval)**; на Турцию перенесён тот же. 95% ДИ — бутстреп по людям (порог фиксирован). Полная таблица со всеми метриками — в csv.

| Модель | Порог (источник) | Набор | Чувств. | Специф. | Сбаланс. accuracy |
|---|---|---|---|---|---|
| BIOT | Юден, валидация NMT | NMT eval | 0,49 (0,38–0,59) | 0,87 (0,80–0,94) | 0,68 |
| BIOT | то же | Турция | 0,00 | 1,00 | 0,50 |
| BIOT | чувств. 80 % | NMT eval | 0,72 (0,62–0,82) | 0,70 (0,61–0,80) | 0,71 |
| BIOT | то же | Турция | 0,02 (0,00–0,07) | 0,99 | 0,50 |
| REVE | Юден, NMT eval* | NMT eval | 0,73 | 0,88 | 0,81 |
| REVE | то же | Турция | 0,04 | 1,00 | 0,52 |
| CBraMod | Юден, NMT eval* | NMT eval | 0,84 | 0,69 | 0,77 |
| CBraMod | то же | Турция | 0,16 | 1,00 | 0,58 |
| DeepEpilepsy | Юден, NMT eval* | NMT eval | 0,94 | 0,69 | 0,82 |
| DeepEpilepsy | то же | Турция | 0,62 (0,48–0,75) | 0,63 (0,52–0,74) | 0,63 |

\* порог по тестовой выборке — оптимистично.

## 9. Выводы

1. **BIOT слабее REVE и сравнима с CBraMod**: ROC AUC 0,82 (0,73–0,89) на Турции и 0,80 (0,73–0,86) на NMT eval. От REVE отстаёт значимо на NMT eval (−0,08), на Турции разница −0,05 с ДИ, включающим 0. От CBraMod на Турции не отличается. Запуски сильно разбросаны на NMT eval (0,71–0,84), на Турции стабильны (0,79–0,81).
2. **Ансамбли с BIOT ничего не дают**: BIOT + REVE ≈ REVE; все четыре ≈ три без BIOT (Турция +0,016, ДИ включает 0). Оценки BIOT сильно согласуются с REVE и CBraMod (0,62–0,66).
3. **Порог, перенесённый с NMT на Турцию, не работает ни у одной модели, кроме DeepEpilepsy**: у BIOT, REVE, CBraMod все эпилептики на Турции получают оценки ниже порога (чувствительность 0–0,16 при специфичности ≈1), хотя ROC AUC 0,82–0,87. Ранжирование сохраняется, калибровка нет: NMT («патология» ~14 % в обучении) и Турция («эпилепсия против здоровых») — разные задачи и разные распределения вероятностей. Для сценария «новая больница» порог, видимо, придётся подбирать заново на небольшой выборке клиники; в этом блокноте это не проверялось.
4. **Связь с амплитудой** у BIOT меньше, чем у REVE (0,25 против 0,31), хотя вход нормирован по каждому каналу: амплитуда напрямую пропадает, косвенные признаки (форма спектра) остаются.
5. **Ограничения**: BIOT предобучена на MGH (рутинные ЭЭГ, не TUH); пересечения предобучения с NMT и Турцией неизвестны; 3 запуска, Турция — 121 человек, ДИ широкие; NMT и Турция — разные задачи; окна 10 с и до 60 на запись, как у REVE; фильтр 0,3–75 Гц + режекция 50 Гц.
6. **Выбор весов**: взят PREST-16 (только MGH, TUH не видела) — подходит, 16 биполярных пар считаются вычитанием из наших 19 каналов. Варианты на 18 каналов не нужны (два лишних канала — «ухо», нет в наших данных), вариант «six datasets» видел обучающие части TUAB/TUEV.

**Затраты:** итоговый блокнот 53,7 мин (обучение 24,3 мин, чтение данных ≈25 мин); первый запуск остановился на проверке (≈20 мин). По квоте Kaggle использовано 15,42 → 17,14 ч (≈1,7 ч с первым запуском; бюджет задачи 1,5 ч **превышен** на ≈0,2 ч, лимит 20 ч соблюдён).

Файлы: `results/biot_finetune_{runs,summary,per_person}.csv`, `results/biot_lr_choice.csv`, `results/biot_ensembles_{summary,per_person}.csv`, `results/biot_threshold_metrics.csv`. В `biot_finetune_per_person.csv` есть вероятности на отложенных 15 % (набор `nmt_val`).